# PARC2026 — Dataset Inventory V1

運営combined datasetまたは検証対象LeRobot datasetについて、**task × episode構成を最初に可視化する**Notebookです。

この段階ではsuccess/collision/replayabilityを推測しません。まずmetadataだけから、task数、episode数、frame数、概算duration、source labelの有無を確定します。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd

ROOT = Path('/content/parc2026')
REPO = ROOT / 'py_AI'
assert REPO.exists(), '00_a100_preflight.ipynb を先に実行してください'
print('repo:', REPO)
print('git:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())

## Dataset rootを指定
`DATASET_ROOT/meta/info.json` と `DATASET_ROOT/meta/episodes.jsonl` が見える場所を指定します。

まずは運営combined datasetを正本にします。Colabへまだ置いていない場合は候補pathを表示して止まります。公開LIBERO系でpipelineだけ先に確認しても構いませんが、その結果を運営dataset inventoryと混同しません。

In [ ]:
DATASET_ROOT = Path(os.environ.get('PARC_DATASET_ROOT', '/content/parc2026/datasets/libero_combined_20hz'))

def find_candidates(base: Path):
    return sorted(p.parent.parent for p in base.glob('**/meta/info.json'))

if not (DATASET_ROOT / 'meta' / 'info.json').exists():
    candidates = find_candidates(ROOT / 'datasets')
    print('configured root not found:', DATASET_ROOT)
    print('detected candidates:')
    for p in candidates[:20]:
        print(' -', p)
    raise FileNotFoundError('PARC_DATASET_ROOT を対象dataset rootへ設定してください')
print('dataset root:', DATASET_ROOT)

## Metadataを確認
ここでfps、features、episode/frame数の宣言値を確認します。

In [ ]:
info = json.loads((DATASET_ROOT / 'meta' / 'info.json').read_text())
print(json.dumps(info, ensure_ascii=False, indent=2)[:12000])

## Inventoryを生成
metadataのみを読み、`episode_inventory.csv` / `task_inventory.csv` / summary JSONを生成します。動画や全Parquetを走査しないため、最初の構造把握を低コストで行えます。

In [ ]:
OUT = ROOT / 'outputs' / 'dataset_inventory_v1'
OUT.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable,
    str(REPO / 'tools/data/build_dataset_inventory.py'),
    '--root', str(DATASET_ROOT),
    '--out', str(OUT),
], check=True)
print('outputs:', OUT)

In [ ]:
summary = json.loads((OUT / 'dataset_inventory_summary.json').read_text())
episodes = pd.read_csv(OUT / 'episode_inventory.csv')
tasks = pd.read_csv(OUT / 'task_inventory.csv')
display(pd.DataFrame([summary]))
display(tasks.head(50))

## Task imbalanceを数値化
Raw / Uniform / Sqrt-balanced の3案を並べます。ここでは採用を決めず、次のcheap ablation用sampling候補を作るだけです。

In [ ]:
import numpy as np

sampling = tasks[['task_name', 'episodes']].copy()
sampling['raw_prob'] = sampling['episodes'] / sampling['episodes'].sum()
sampling['uniform_prob'] = 1.0 / len(sampling)
sqrt_n = np.sqrt(sampling['episodes'].clip(lower=1))
sampling['sqrt_balanced_prob'] = sqrt_n / sqrt_n.sum()
sampling = sampling.sort_values('episodes', ascending=False)
sampling.to_csv(OUT / 'task_sampling_candidates.csv', index=False)
display(sampling.head(50))
print('max/min episode ratio:', sampling['episodes'].max() / max(1, sampling['episodes'].min()))
print('saved:', OUT / 'task_sampling_candidates.csv')

## 分布を見る
task別episode数とepisode長を見て、極端な偏り・長さの外れ値を確認します。これはquality filteringの閾値そのものではありません。

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
tasks['episodes'].hist(bins=min(50, max(10, len(tasks)//2)))
plt.xlabel('episodes per task')
plt.ylabel('task count')
plt.title('Task episode-count distribution')
plt.show()

if 'frames' in episodes and episodes['frames'].notna().any():
    plt.figure(figsize=(10, 4))
    episodes['frames'].dropna().hist(bins=50)
    plt.xlabel('frames per episode')
    plt.ylabel('episode count')
    plt.title('Episode length distribution')
    plt.show()

## 次のDataset Factoryへの入力
このNotebookで確定するのは **V0 Organizer Rawの構造** です。

次に使う成果物:
- `episode_inventory.csv` — 1行=1 episode
- `task_inventory.csv` — taskごとのepisode/frame/duration
- `task_sampling_candidates.csv` — raw / uniform / sqrt-balanced候補
- `dataset_inventory_summary.json` — dataset root/fps/counts

次段ではStatic Quality Analyzerを追加して movement/path/jerk/idle をepisode単位で計測し、V1 Clean候補を作ります。success/collision/replayabilityはraw simulator stateが確保できる場合だけReplay Validatorで追加します。